# Preprocess

In [ ]:
from preprocess import run_full_pipeline
resultats = run_full_pipeline(r"C:\TSQ\data\Projet_final_DESU\data2.csv")

df = resultats["df"]
df_PS = resultats["df_PS"]
df_PD = resultats["df_PD"]

df_SD = resultats["df_SD"]
df_SD_PS = resultats["df_SD_PS"]
df_SD_PD = resultats["df_SD_PD"]

df_BAI = resultats["df_BAI"]
df_TSQ = resultats["df_TSQ"]
df_TSQ_PD = resultats["df_TSQ_PD"]

df_EP = resultats["df_EP"]
df_EP_PD = resultats["df_EP_PD"]

df_16 = resultats["df_16"]
df_16_PD = resultats["df_16_PD"]
echelle1 = resultats["echelle1"]
echelle2 = resultats["echelle2"]

df_sub = resultats["df_sub"]

df_ZTPI = resultats["df_ZTPI"]
df_ZTPI_PD = resultats["df_ZTPI_PD"]


# Création de groupes au sein de l'échelle TSQ: 
* 0 = personne saine 
* 1 = personne se déclarant avec diagnostique

In [ ]:
import pandas as pd
df_TSQ = df_TSQ.copy()
df_TSQ["group"] = 0

df_TSQ_PD = df_TSQ_PD.copy()
df_TSQ_PD["group"] = 1

df_all = pd.concat([df_TSQ, df_TSQ_PD])
group = df_all["group"]          
df_all = df_all.drop(columns="group")   


On renomme les colonnes : intitulé des questions --> TSQ1 +n

In [ ]:
dfs = [df_TSQ, df_TSQ_PD, df_all]

for df in dfs:
    
    cols_tsq_brutes = df.columns[0:40]

    # On renomme les 40 items : TSQ1 → TSQ40
    new_names = [f"TSQ{i}" for i in range(1, 41)]
    df.rename(columns=dict(zip(cols_tsq_brutes, new_names)), inplace=True)

    df[new_names] = df[new_names].apply(
        lambda col: col.astype(str).str.extract(r"(\d+)")[0].astype(float)
    )


# Correlation des 40x40 items de la TSQ + greedy glouton pour la redondance des corrélations

In [ ]:
from correl_TSQ import compute_tsq_corr

compute_tsq_corr(df_all, items=[f"TSQ{i}" for i in range(1, 41)], title="Matrice de correlation 40x40 - tous les participants")

from correl_TSQ import reduce_correlated_features
items = [f"TSQ{i}" for i in range(1, 41)]
selected_items = reduce_correlated_features(df_all, items, threshold=0.65)

print("Items conservés :", selected_items)
print("Items retirés :", [i for i in items if i not in selected_items])



# VIF pour voir la multicollinéarité

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

X = df_all[items].dropna()
vif_data = pd.DataFrame()
vif_data['item'] = X.columns
vif_data['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print(vif_data.sort_values('VIF', ascending=False))

In [ ]:
colonnes_a_exclure = ['TSQ5', 'TSQ9', 'TSQ22', 'TSQ25', 'TSQ39', 'score_total']  
df_red_sains = df_TSQ.drop(columns=colonnes_a_exclure)
df_red_pd = df_TSQ_PD.drop(columns=colonnes_a_exclure)

# methodes de la silhouette score, du coude, indice de Davies-Bouldin, indice de Calinski-Harabasz pour déterminer combien de k doit on prendre pour le K-means (étape d'après)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import matplotlib.pyplot as plt
import numpy as np

X = df_red_sains.values  

k_values = range(2, 11)
inerties = []
silhouettes = []
davies_bouldin = []
calinski_harabasz = []

for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X)

    inerties.append(kmeans.inertia_)
    silhouettes.append(silhouette_score(X, labels))
    davies_bouldin.append(davies_bouldin_score(X, labels))
    calinski_harabasz.append(calinski_harabasz_score(X, labels))

    print(f"k={k} | inertie={kmeans.inertia_:.1f} | silhouette={silhouettes[-1]:.3f} "
          f"| davies-bouldin={davies_bouldin[-1]:.3f} | calinski-harabasz={calinski_harabasz[-1]:.1f}")


resultats_k = pd.DataFrame({
    "k": list(k_values),
    "inertie": inerties,
    "silhouette": silhouettes,
    "davies_bouldin": davies_bouldin,
    "calinski_harabasz": calinski_harabasz,
})

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

axes[0, 0].plot(k_values, inerties, marker="o")
axes[0, 0].set_title("Méthode du coude (inertie)")
axes[0, 0].set_xlabel("k")
axes[0, 0].set_ylabel("Inertie")

axes[0, 1].plot(k_values, silhouettes, marker="o", color="orange")
axes[0, 1].set_title("Score de silhouette (plus haut = mieux)")
axes[0, 1].set_xlabel("k")
axes[0, 1].set_ylabel("Silhouette")

axes[1, 0].plot(k_values, davies_bouldin, marker="o", color="green")
axes[1, 0].set_title("Indice de Davies-Bouldin (plus bas = mieux)")
axes[1, 0].set_xlabel("k")
axes[1, 0].set_ylabel("Davies-Bouldin")

axes[1, 1].plot(k_values, calinski_harabasz, marker="o", color="red")
axes[1, 1].set_title("Indice de Calinski-Harabasz (plus haut = mieux)")
axes[1, 1].set_xlabel("k")
axes[1, 1].set_ylabel("Calinski-Harabasz")

for ax in axes.flat:
    ax.set_xticks(list(k_values))

plt.tight_layout()
plt.show()

# Meilleur k selon chaque indice
print("\nMeilleur k selon chaque indice :")
print(f"  Silhouette (max)        : k={resultats_k.loc[resultats_k['silhouette'].idxmax(), 'k']} "
      f"(score={resultats_k['silhouette'].max():.3f})")
print(f"  Davies-Bouldin (min)    : k={resultats_k.loc[resultats_k['davies_bouldin'].idxmin(), 'k']} "
      f"(score={resultats_k['davies_bouldin'].min():.3f})")
print(f"  Calinski-Harabasz (max) : k={resultats_k.loc[resultats_k['calinski_harabasz'].idxmax(), 'k']} "
      f"(score={resultats_k['calinski_harabasz'].max():.1f})")

resultats_k

Donc on va prendre k = 2 pour le clustering

# UMAP sur tous les participants sains sur le df reduit (35 items)

In [ ]:
from umap_clustering import run_umap_clustering

result = run_umap_clustering(
    df_red_sains,
    n_clusters=2,
    random_state=42,
    group=group,
)


df_clust = result.df_clust         
clusters = result.clusters          
proj_2d = result.proj_2d            
proj_3d = result.proj_3d            
fig_2d, fig_3d = result.fig_2d, result.fig_3d

# Split train/test des participants sains + entrainement des modeles classifier (4 différents) sur le train et test pour voir si l'algo me classe bien dans le bon cluster (par rapport au UMAP) les participants. Grid search pour trouver le meilleur modele

In [ ]:
from sklearn.model_selection import train_test_split
from umap_clustering import run_umap_clustering
from machinelearning import train_and_compare_classifiers, apply_to_patients

# resultat du umap
result = run_umap_clustering(
    df_red_sains, n_clusters=2, random_state=42, group=group, show=False,
)

# reprend les resultats umap pour voir ou sont repartis les participants cluster 0 ou 1
X_all = result.proj_2d
y_all = result.clusters

# Split train/test SUR les projections/labels déjà calculés 
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all,
    test_size=0.25,
    random_state=42,
    stratify=y_all,  # garde la proportion des clusters
)

# Entraîne et compare les classifieurs
results, summary, best_model_name, best_model = train_and_compare_classifiers(
    X_train, y_train, X_test, y_test, scoring="balanced_accuracy"
)

Clairement overfit 

# Réprésentation de la prédiction de la répartition des patients sur les clusters 0 ou 1 dans l'espace UMAP des participants sains

In [ ]:
X_patients_2d = result.umap_2d_model.transform(df_red_pd.values) #  Projete les patients dans l'espace UMAP appris sur les sains (transform, pas fit)

y_pred_patients = best_model.predict(X_patients_2d) # Applique le best modèle de classif pour prédire leur "cluster" 

df_patients_clust = df_red_pd.copy()
df_patients_clust['cluster_pred'] = y_pred_patients

print(df_patients_clust['cluster_pred'].value_counts().sort_index())

In [ ]:
from sklearn.model_selection import train_test_split
X_all = result.proj_2d
y_all = result.clusters
idx_all = df_red_sains.index

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X_all, y_all, idx_all,
    test_size=0.25,
    random_state=42,
    stratify=y_all,
)


X_patients_umap = result.umap_2d_model.transform(df_red_pd.values)

import plotly.express as px

hover_train = [f"Participant {i}" for i in idx_train]
hover_test  = [f"Participant {i}" for i in idx_test]
hover_diag  = [f"Participant {i}" for i in df_red_pd.index]

df_plot = pd.concat([
    pd.DataFrame({
        "x": X_train[:, 0], "y": X_train[:, 1],
        "cluster": y_train.astype(str),
        "type": "Sains (train)",
        "participant": hover_train,
    }),
    pd.DataFrame({
        "x": X_test[:, 0], "y": X_test[:, 1],
        "cluster": y_test.astype(str),
        "type": "Sains (test)",
        "participant": hover_test,
    }),
    pd.DataFrame({
        "x": X_patients_umap[:, 0], "y": X_patients_umap[:, 1],
        "cluster": df_patients_clust['cluster_pred'].astype(str),
        "type": "Diag",
        "participant": hover_diag,
    }),
], ignore_index=True)

fig = px.scatter(
    df_plot, x="x", y="y",
    color="cluster", symbol="type",
    color_discrete_map={"0": '#FC6955', "1": "#A777F1" },
    symbol_map={"Sains (train)": "circle", "Sains (test)": "circle-open", "Diag": "triangle-up"},
    opacity=0.7,
    hover_name="participant",
    title="Patients projetés dans l'espace UMAP crée sur les participants sains",
)

for trace in fig.data:
    if "Diag" in trace.name:
        trace.marker.line = dict(width=1, color="black")

fig.show(renderer="browser")

# 2 ème partie d'analyse
* creation de chaque df pour chaque groupes d'items
* UMAP + Clustering sur participants sains pour chaque df 
* ajouter à la boucle : Split train/test des participants sains + Classifier (4 modeles) avec GridSearch pour chaque df
* visualisation du best model sur le groupe de participants diag

In [ ]:
#Core items

items_core_sz = ['TSQ2','TSQ4','TSQ7','TSQ8','TSQ10','TSQ11','TSQ12','TSQ15','TSQ19','TSQ20','TSQ21','TSQ24','TSQ30','TSQ31','TSQ34','TSQ40']
items_core_md = ['TSQ9','TSQ14','TSQ17','TSQ25','TSQ28','TSQ29','TSQ36']
items_core_ma = ['TSQ22','TSQ32','TSQ35','TSQ39']
items_core_an = ['TSQ1','TSQ5','TSQ6','TSQ9','TSQ13','TSQ16','TSQ17','TSQ20','TSQ23','TSQ33','TSQ35','TSQ37','TSQ38','TSQ39']
items_core_pt = ['TSQ2','TSQ3','TSQ5','TSQ6','TSQ17','TSQ18','TSQ26','TSQ30','TSQ37']

#Periphery items
items_per_sz = ['TSQ1','TSQ3','TSQ5','TSQ6','TSQ13','TSQ14','TSQ23','TSQ27','TSQ35','TSQ38','TSQ39']
items_per_md = ['TSQ1','TSQ2','TSQ8','TSQ12','TSQ16','TSQ18','TSQ19','TSQ20','TSQ23','TSQ27','TSQ31']
items_per_ma = ['TSQ3','TSQ7','TSQ9','TSQ19','TSQ21','TSQ30','TSQ34','TSQ37','TSQ38']
items_per_an = ['TSQ2','TSQ3','TSQ12','TSQ18','TSQ19','TSQ22','TSQ25','TSQ27','TSQ32']
items_per_pt = ['TSQ1','TSQ9','TSQ12','TSQ13','TSQ14','TSQ16','TSQ19','TSQ20','TSQ23','TSQ24','TSQ25','TSQ27','TSQ28','TSQ31','TSQ33','TSQ35','TSQ36','TSQ38','TSQ39','TSQ40']

#Combined Core+Periphery items 
items_tot_sz = items_core_sz + items_per_sz
items_tot_md = items_core_md + items_per_md
items_tot_ma = items_core_ma + items_per_ma
items_tot_an = items_core_an + items_per_an
items_tot_pt = items_core_pt + items_per_pt

In [ ]:
# --- Core items ---
df_core_ax   = df_all[items_core_an].copy()
df_core_sz   = df_all[items_core_sz].copy()
df_core_mdd  = df_all[items_core_md].copy()
df_core_man  = df_all[items_core_ma].copy()
df_core_ptsd = df_all[items_core_pt].copy()

# --- Periphery items ---
df_periph_ax   = df_all[items_per_an].copy()
df_periph_sz   = df_all[items_per_sz].copy()
df_periph_mdd  = df_all[items_per_md].copy()
df_periph_man  = df_all[items_per_ma].copy()
df_periph_ptsd = df_all[items_per_pt].copy()

# --- Core + Periphery combinés ---
df_tot_ax   = df_all[items_tot_an].copy()
df_tot_sz   = df_all[items_tot_sz].copy()
df_tot_mdd  = df_all[items_tot_md].copy()
df_tot_man  = df_all[items_tot_ma].copy()
df_tot_ptsd = df_all[items_tot_pt].copy()

#dico
sous_echelles = {
    "core_ax": df_core_ax, "core_sz": df_core_sz, "core_mdd": df_core_mdd,
    "core_man": df_core_man, "core_ptsd": df_core_ptsd,
    "periph_ax": df_periph_ax, "periph_sz": df_periph_sz, "periph_mdd": df_periph_mdd,
    "periph_man": df_periph_man, "periph_ptsd": df_periph_ptsd,
}

In [ ]:
# --- Core items ---
df_core_ax   = df_TSQ[items_core_an].copy()
df_core_sz   = df_TSQ[items_core_sz].copy()
df_core_mdd  = df_TSQ[items_core_md].copy()
df_core_man  = df_TSQ[items_core_ma].copy()
df_core_ptsd = df_TSQ[items_core_pt].copy()

# --- Periphery items ---
df_periph_ax   = df_TSQ[items_per_an].copy()
df_periph_sz   = df_TSQ[items_per_sz].copy()
df_periph_mdd  = df_TSQ[items_per_md].copy()
df_periph_man  = df_TSQ[items_per_ma].copy()
df_periph_ptsd = df_TSQ[items_per_pt].copy()

# --- Core + Periphery combinés ---
df_tot_ax   = df_TSQ[items_tot_an].copy()
df_tot_sz   = df_TSQ[items_tot_sz].copy()
df_tot_mdd  = df_TSQ[items_tot_md].copy()
df_tot_man  = df_TSQ[items_tot_ma].copy()
df_tot_ptsd = df_TSQ[items_tot_pt].copy()

#dico
sous_echelles = {
    "core_ax": df_core_ax, "core_sz": df_core_sz, "core_mdd": df_core_mdd,
    "core_man": df_core_man, "core_ptsd": df_core_ptsd,
    "periph_ax": df_periph_ax, "periph_sz": df_periph_sz, "periph_mdd": df_periph_mdd,
    "periph_man": df_periph_man, "periph_ptsd": df_periph_ptsd,
}
# --- Core items ---
df_core_ax_PD   = df_TSQ_PD[items_core_an].copy()
df_core_sz_PD   = df_TSQ_PD[items_core_sz].copy()
df_core_mdd_PD  = df_TSQ_PD[items_core_md].copy()
df_core_man_PD  = df_TSQ_PD[items_core_ma].copy()
df_core_ptsd_PD = df_TSQ_PD[items_core_pt].copy()

# --- Periphery items ---
df_periph_ax_PD   = df_TSQ_PD[items_per_an].copy()
df_periph_sz_PD   = df_TSQ_PD[items_per_sz].copy()
df_periph_mdd_PD  = df_TSQ_PD[items_per_md].copy()
df_periph_man_PD  = df_TSQ_PD[items_per_ma].copy()
df_periph_ptsd_PD = df_TSQ_PD[items_per_pt].copy()

# --- Core + Periphery combinés ---
df_tot_ax_PD   = df_TSQ_PD[items_tot_an].copy()
df_tot_sz_PD   = df_TSQ_PD[items_tot_sz].copy()
df_tot_mdd_PD  = df_TSQ_PD[items_tot_md].copy()
df_tot_man_PD  = df_TSQ_PD[items_tot_ma].copy()
df_tot_ptsd_PD = df_TSQ_PD[items_tot_pt].copy()

#dico
sous_echelles_PD = {
    "core_ax_PD": df_core_ax_PD, "core_sz_PD": df_core_sz_PD, "core_mdd_PD": df_core_mdd_PD,
    "core_man_PD": df_core_man_PD, "core_ptsd_PD": df_core_ptsd_PD,
    "periph_ax_PD": df_periph_ax_PD, "periph_sz_PD": df_periph_sz_PD, "periph_mdd_PD": df_periph_mdd_PD,
    "periph_man_PD": df_periph_man_PD, "periph_ptsd_PD": df_periph_ptsd_PD,
}

In [ ]:
from correl_TSQ import compute_tsq_corr, reduce_correlated_features

resultats_corr = {}

for nom, df_subset in sous_echelles.items():
    print(f"\n{'='*25} {nom} {'='*25}")

    items = list(df_subset.columns)

    
    compute_tsq_corr(df_all, items=items, title=nom)

    selected_items = reduce_correlated_features(df_all, items, threshold=0.65)
    dropped_items = [i for i in items if i not in selected_items]

    print(f"Items à conserver ({len(selected_items)}/{len(items)}) :", selected_items)
    print(f"Items à retirer ({len(dropped_items)}) :", dropped_items)

    resultats_corr[nom] = {
        "items_initiaux": items,
        "selected_items": selected_items,
        "dropped_items": dropped_items,
    }

In [ ]:
sous_echelles_red = {}
sous_echelles_PD_red = {}

for nom, df_sains in sous_echelles.items():
    dropped = resultats_corr[nom]["dropped_items"] + ["score_total"]

    df_sains_red = df_sains.drop(columns=dropped, errors="ignore")
    df_pd_red = sous_echelles_PD[f"{nom}_PD"].drop(columns=dropped, errors="ignore")

    sous_echelles_red[nom] = df_sains_red
    sous_echelles_PD_red[f"{nom}_PD"] = df_pd_red

    print(f"{nom}: {df_sains.shape[1]} → {df_sains_red.shape[1]} items "
          f"(retirés: {dropped if dropped else 'aucun'})")

# Boucle sur chaque sous echelle : 
* UMAP sur participants sains +
*  clustering (2 groupes) + 
* train/test sur classifier 
* --> best model à tester sur les participants avec diag --> voir ou sont classés la pop diag concernée par rapport aux sains et aux autres patho

* ex: patients depressifs sont-ils dans le meme cluster/ le + symptomato pour le df core depression? 

In [ ]:
results, summary, best_model_name, best_model = train_and_compare_classifiers(
    X_train, y_train, X_test, y_test, scoring="balanced_accuracy"
)

In [ ]:
from umap_clustering import run_umap_clustering
from machinelearning import train_and_compare_classifiers, apply_to_patients

resultats_sous_echelles = {}

for nom, df_sains in sous_echelles_red.items():
    print(f"\n{'='*25} {nom} {'='*25}")

    df_patients = sous_echelles_PD_red[f"{nom}_PD"]

    # UMAP + KMeans fit sur TOUS les participants sains 
    result = run_umap_clustering(
        df_sains, n_clusters=2, random_state=42, group=group, show=False,
    )

    # Split train/test des sains SUR les projections/labels déjà calculés (pour le classifieur)
    X_all = result.proj_2d
    y_all = result.clusters
    idx_all = df_sains.index

    X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
        X_all, y_all, idx_all,
        test_size=0.25,
        random_state=42,
        stratify=y_all,
    )

    # Entraîner et comparer les classifieurs
    results, summary, best_model_name, best_model = train_and_compare_classifiers(
        X_train, y_train, X_test, y_test,
        scoring="balanced_accuracy",
        plot_curves=True,
    )

    # Application aux patients (transform UMAP + predict, jamais refit)
    df_patients_pred = apply_to_patients(
        best_model=best_model,
        umap_model=result.umap_2d_model,
        df_red_pd=df_patients,
    )

    resultats_sous_echelles[nom] = {
        "result": result,             
        "X_train": X_train, "X_test": X_test,
        "y_train": y_train, "y_test": y_test,
        "idx_train": idx_train, "idx_test": idx_test,
        "results": results,
        "summary": summary,
        "best_model_name": best_model_name,
        "best_model": best_model,
        "df_patients_pred": df_patients_pred,
    }

    print(f"Meilleur modèle pour {nom} : {best_model_name}")

clairement overfit le mod ne foncctionne pas : surtout pour core manie et periph mdd. 

# Récap des meilleurs modeles trouvés pour chq sous echelle

In [ ]:
recap = []
for nom, r in resultats_sous_echelles.items():
    best_row = r["summary"].loc[r["best_model_name"]]
    patient_counts = r["df_patients_pred"]["predicted_cluster"].value_counts().sort_index()
    recap.append({
        "sous_echelle": nom,
        "best_model": r["best_model_name"],
        "cv_score": round(best_row["CV score"], 3),
        "test_score": round(best_row["Test score"], 3),
        "patients_cluster_0": patient_counts.get(0, 0),
        "patients_cluster_1": patient_counts.get(1, 0),
    })

df_recap = pd.DataFrame(recap).sort_values("test_score", ascending=False)
df_recap

# Visualisation de la classification des participants avec diag dans l'espace UMAP

In [ ]:
import pandas as pd
import plotly.express as px

for nom, r in resultats_sous_echelles.items():
    result = r["result"]
    df_patients_pred = r["df_patients_pred"]
    df_patients = sous_echelles_PD_red[f"{nom}_PD"]

    X_patients_umap = result.umap_2d_model.transform(df_patients.values)

    hover_train = [f"Participant {i}" for i in r["idx_train"]]
    hover_test  = [f"Participant {i}" for i in r["idx_test"]]
    hover_diag  = [f"Participant {i}" for i in df_patients.index]

    df_plot = pd.concat([
        pd.DataFrame({
            "x": r["X_train"][:, 0], "y": r["X_train"][:, 1],
            "cluster": r["y_train"].astype(str),
            "type": "Sains (train)",
            "participant": hover_train,
        }),
        pd.DataFrame({
            "x": r["X_test"][:, 0], "y": r["X_test"][:, 1],
            "cluster": r["y_test"].astype(str),
            "type": "Sains (test)",
            "participant": hover_test,
        }),
        pd.DataFrame({
            "x": X_patients_umap[:, 0], "y": X_patients_umap[:, 1],
            "cluster": df_patients_pred["predicted_cluster"].astype(str),
            "type": "Diag",
            "participant": hover_diag,
        }),
    ], ignore_index=True)

    fig = px.scatter(
        df_plot, x="x", y="y",
        color="cluster", symbol="type",
        color_discrete_map={"0": "#A777F1", "1": "#FC6955"},
        symbol_map={"Sains (train)": "circle", "Sains (test)": "circle-open", "Diag": "triangle-up"},
        opacity=0.7,
        hover_name="participant",
        title=f"{nom} — Sains (train/test) et patients projetés dans l'espace UMAP (fit sur tous les sains)",
    )

    for trace in fig.data:
        if "Diag" in trace.name:
            trace.marker.line = dict(width=1, color="black")

    fig.show(renderer="browser")

# Regardons si le profil des participants avec diag mis dans le cluster 1 correspond à leur diag en fonction des sous echelles

In [ ]:
col_name = df_PD.columns[18]
print(f"Colonne diag : {col_name}")

diag = df_PD[col_name]
repartition_patients = {}

for nom, r in resultats_sous_echelles.items():
    df_patients_pred = r["df_patients_pred"]

    # Items complets (non réduits) de la sous-échelle -> calcul du score normalisé (Northoff)
    df_items_nom = sous_echelles_PD[f"{nom}_PD"]
    n_items = df_items_nom.shape[1]
    score_nom = df_items_nom.sum(axis=1) / (10 * n_items)

    cluster_0_idx = df_patients_pred[df_patients_pred["predicted_cluster"] == 0].index
    cluster_1_idx = df_patients_pred[df_patients_pred["predicted_cluster"] == 1].index

    cluster_0 = list(zip(
        cluster_0_idx,
        diag.reindex(cluster_0_idx),
        score_nom.reindex(cluster_0_idx),
    ))
    cluster_1 = list(zip(
        cluster_1_idx,
        diag.reindex(cluster_1_idx),
        score_nom.reindex(cluster_1_idx),
    ))

    repartition_patients[nom] = {
        "cluster_0": cluster_0,
        "cluster_1": cluster_1,
    }

    print(f"\n{'='*25} {nom} {'='*25}")
    print(f"Cluster 0 ({len(cluster_0)} patients) :")
    for participant, d, s in cluster_0:
        print(f"  Participant {participant} → {d} | score {nom} = {s:.3f}")
    print(f"Cluster 1 ({len(cluster_1)} patients) :")
    for participant, d, s in cluster_1:
        print(f"  Participant {participant} → {d} | score {nom} = {s:.3f}")


lignes = []
for nom, r in repartition_patients.items():
    for cl in ["cluster_0", "cluster_1"]:
        scores = pd.Series([s for _, _, s in r[cl]], dtype=float)
        lignes.append({
            "sous_echelle": nom,
            "cluster": cl,
            "n": len(scores),
            "moyenne": scores.mean(),
            "min": scores.min(),
            "max": scores.max(),
        })

stats_clusters_PD = pd.DataFrame(lignes).round(3)
print(stats_clusters_PD.to_string(index=False))

Finalement le cluster 0 ou 1 change : le + ou - symptomatique selon les df. Globalement apres analyse: les items spé de la /// sont représentatifs de ... 
Attention, les modèles ont surappris, on a pas assez d'echantillon et encore moins de patients! 

# SFS - Sequential Feature Selection - backward

Boucle pour chaque type de patho (sauf comorbidité qui ne faisait pas partie de l'echelle) : 
Par exemple, on prend les patients MDD qu'on renomme "1" et tous les controles "0", sur les 40 items. On entraine un classifier sur tout (split train/test sur les 2 groupes )avec cv = stratifiedKfold car groupes non equilibrés. Puis on rgerade si classifier capable de classer 0 et 1. Va me sortir les items spé des patients MDD avec le SFS. A comparer avec ceux établis par Northoff 

In [ ]:
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import balanced_accuracy_score, classification_report
import numpy as np

# Groupes diagnostiques 
sz    = df_PD.iloc[[10]]
man   = df_PD.iloc[[4, 21, 22, 26, 29, 30]]
ax    = df_PD.iloc[[1, 12, 13, 15, 24]]
dep   = df_PD.iloc[[2, 3, 5, 6, 7, 8, 9, 16, 18, 19, 28]]
comor = df_PD.iloc[[0, 11, 14, 17, 20, 23, 25, 27]]

# Mapping des noms des sous-échelle -> groupe patients ciblé + items core/periph prédéfinis
diag_map = {
    "mdd":  {"idx_target": dep.index,  "items_core": items_core_md, "items_periph": items_per_md},
    "sz":   {"idx_target": sz.index,   "items_core": items_core_sz, "items_periph": items_per_sz},
    "man":  {"idx_target": man.index,  "items_core": items_core_ma, "items_periph": items_per_ma},
    "ax":   {"idx_target": ax.index,   "items_core": items_core_an, "items_periph": items_per_an},
}

items_40 = [f"TSQ{i}" for i in range(1, 41)]
df_healthy_40 = df_TSQ[items_40]   

resultats_sfs_diag = {}

for nom, cfg in diag_map.items():
    print(f"\n{'='*25} {nom.upper()} {'='*25}")

    idx_target = cfg["idx_target"]
    idx_common = df_TSQ_PD.index.intersection(idx_target)
    df_patients_target = df_TSQ_PD.loc[idx_common, items_40]

    print(f"{len(df_patients_target)} patients '{nom}' trouvés / {len(idx_target)} attendus")

    if len(df_patients_target) < 5:
        print(f"Trop peu de patients '{nom}' ({len(df_patients_target)}) pour un split fiable, on passe.")
        continue

    # Dataset binaire : 0 = sain, 1 = diag ciblé, sur les 40 items
    X_full = pd.concat([df_healthy_40, df_patients_target], axis=0)
    y_full = np.array([0] * len(df_healthy_40) + [1] * len(df_patients_target))
    idx_full = X_full.index

    X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
        X_full, y_full, idx_full,
        test_size=0.25, random_state=42, stratify=y_full,
    )

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    clf = RandomForestClassifier(
        n_estimators=300, max_depth=5, min_samples_leaf=5,
        class_weight="balanced", random_state=42,
    )

    baseline_scores = cross_val_score(clf, X_train, y_train, cv=cv, scoring="balanced_accuracy")
    print(f"Score CV baseline (40 items) : {baseline_scores.mean():.3f} ± {baseline_scores.std():.3f}")

    sfs = SequentialFeatureSelector(
        clf, direction="backward", n_features_to_select="auto",#essayer de changer 5 par ex
        tol=-0.005, scoring="balanced_accuracy", cv=cv, n_jobs=-1,
    )
    sfs.fit(X_train, y_train)

    selected_items = list(X_train.columns[sfs.get_support()])
    print(f"Items sélectionnés par SFS ({len(selected_items)}/40) :", selected_items)

    clf.fit(X_train[selected_items], y_train)
    y_pred = clf.predict(X_test[selected_items])
    test_score = balanced_accuracy_score(y_test, y_pred)
    print(f"Score test (items SFS) : {test_score:.3f}")
    print(classification_report(y_test, y_pred))

    # Comparaison avec les items core/periphery prédéfinis par Northoff
    items_core = set(cfg["items_core"])
    items_periph = set(cfg["items_periph"])
    selected_set = set(selected_items)

    overlap_core = selected_set & items_core
    overlap_periph = selected_set & items_periph
    overlap_neither = selected_set - items_core - items_periph

    print(f"Overlap avec items_core_{nom} : {sorted(overlap_core)} ({len(overlap_core)}/{len(items_core)})")
    print(f"Overlap avec items_periph_{nom} : {sorted(overlap_periph)} ({len(overlap_periph)}/{len(items_periph)})")
    print(f"Items SFS hors core/periph prédéfinis : {sorted(overlap_neither)}")

    resultats_sfs_diag[nom] = {
        "n_patients": len(df_patients_target),
        "selected_items": selected_items,
        "baseline_cv_score": baseline_scores.mean(),
        "test_score_sfs": test_score,
        "overlap_core": overlap_core,
        "overlap_periph": overlap_periph,
        "idx_train": idx_train, "idx_test": idx_test,
        "sfs": sfs, "clf": clf,
    }

n'affichera pas pour les sz ni ptsd par exemple car trop peu de patients: ne sert a rien de split train/test

In [ ]:
recap_sfs_diag = pd.DataFrame({
    nom: {
        "n_patients": r["n_patients"],
        "n_items_selected": len(r["selected_items"]),
        "baseline_cv": round(r["baseline_cv_score"], 3),
        "test_score": round(r["test_score_sfs"], 3),
        "n_overlap_core": len(r["overlap_core"]),
        "n_overlap_periph": len(r["overlap_periph"]),
    }
    for nom, r in resultats_sfs_diag.items()
}).T

recap_sfs_diag

# RFECV - Recursive feature elimination with cross-validation to select features.

In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import rfecv as prg

resultats_analyse = prg.run_full_analysis(
    df_PD=df_PD,
    df_sains_40=df_TSQ,
    df_red_sains=df_red_sains,
    df_pd_40=df_TSQ_PD,
    df_red_pd=df_red_pd,
    group=group,
    items_core_sz=items_core_sz,
    items_core_ma=items_core_ma,
    items_core_an=items_core_an,
    items_core_md=items_core_md,
)
recaps = resultats_analyse["recaps"]